<div style="position:relative; width:100%; height:200px;">
  <img src="https://raw.githubusercontent.com/stefanlessmann/VHB_ProDoc_ML/master/banner-nb.png" style="width:100%; object-fit:cover;" alt="ProDok-MachineLearning-Banner">
  <div style="
      position:absolute;
      left:4%;
      top:50%;
      transform:translateY(-50%);
      font-size:3.2vw;
      font-weight:750;
      color:#1f2a44;">
    ProDok – Machine Learning
  </div>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stefanlessmann/VHB_ProDoc_ML/blob/master/P.I.2.cs_benchmark.ipynb)

# P.I.2 Machine Learning for Credit Risk Model Benchmarking
The practice sessions complement the lectures and provide hands-on experience with the concepts covered in the course.
This session focuses on classic machine learning algorithms and practices. We will continue working with the credit risk analytics case, and benchmark alternative supervised learning algorithms to default prediction. In this scope, we will revisit data organization principles, classification model evaluation, and hyperparameter tuning. 
As in the previous practice session, the available time does not permit manual coding or extensive code reviews. We will provide coding demos for selected parts and otherwise rely on LLMs to generate the codes we need. The focus of the practice session is on prompt engineering and discussing ML outputs.  


# Data Preparation Reloaded

**Context:** 
We received a new sample of credit risk data. The sample comprises the same features and was taken 6 month after the first 100k batch was gathered. We developed Python codes to ready the data for analysis. However, the code was AI generated and not designed for reusability. Our first task is to *refactor* the data preparation code to ensure can use for the first and the second batch of data, as well as future batches yet to come. 

In [1]:
# Importing standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

url_100k = "https://raw.githubusercontent.com/stefanlessmann/VHB_ProDoc_ML/master/credit_data_100k.csv"
url_25k = "https://raw.githubusercontent.com/stefanlessmann/VHB_ProDoc_ML/master/credit_data_25k.csv"

df_100 = pd.read_csv(url_100k)
df_25 = pd.read_csv(url_25k)

## Task: 
Use the code from `P.I.1.data_exploaration.ipynb` to create a reusable data preparation function. Apply your function to create *ready-for-modeling* versions of both datasets. Store the results in variables `dfDevelopment` and `dfHoldout`, respectively. 

In general, LLMs are good at refactoring code. Put our generated data preparation code into a prompt and instructing an LLM to refactor is should work just fine. One the other hand, all we need to do is wrapping up our previous code in a function. Decide freely if you want to approach the task by prompting an LLM or by coding the data preparation function yourself.

>Note that this exercise focuses on classification. Therefore, make sure to remove the `LGD` column from the data frames to prevent data leakage.




In [2]:
# Place for your data preparation function
def prepare_credit_data(
    df: pd.DataFrame,
    winsor_limits=(0.01, 0.99),
    skew_threshold=1.0,
    add_missing_indicators=True,
    add_log_features=True,
    verbose=True
):
    """
    Prepare credit risk dataset for modeling.

    Parameters
    ----------
    df : pd.DataFrame
        Raw input dataframe following the defined data dictionary structure.
    winsor_limits : tuple(float, float)
        Lower/upper quantiles for winsorization of numeric predictors.
    skew_threshold : float
        Skewness threshold above which log1p features are added.
    add_missing_indicators : bool
        Whether to create missing indicator columns for numeric variables.
    add_log_features : bool
        Whether to create log1p features for skewed positive variables.
    verbose : bool
        Whether to print diagnostics.

    Returns
    -------
    dict with:
        df_model : modeling-ready dataframe (predictors + targets)
        predictors : list of predictor column names
        targets : list of target column names
        metadata : dict with preprocessing details
    """

    df_prep = df.copy(deep=True)

    # ---------------------------
    # 1) DEFINE ROLES
    # ---------------------------
    target_cols = ["default_12m", "lgd"]
    binary_cols = ["default_12m", "has_bank_account", "secured"]
    categorical_cols = ["employment_type", "housing_status", "region", "loan_purpose"]
    numeric_cols = [
        "age", "income", "years_at_job", "years_at_address",
        "loan_amount", "dti",
        "num_inquiries_6m", "num_delinquencies_24m",
        "num_collections", "card_utilization"
    ]

    present = set(df_prep.columns)
    target_cols = [c for c in target_cols if c in present]
    binary_cols = [c for c in binary_cols if c in present]
    categorical_cols = [c for c in categorical_cols if c in present]
    numeric_cols = [c for c in numeric_cols if c in present]

    # ---------------------------
    # 2) TYPE ENFORCEMENT
    # ---------------------------
    # Binary handling
    for c in binary_cols:
        if df_prep[c].dtype == "bool":
            df_prep[c] = df_prep[c].astype(int)

        if df_prep[c].dtype == "object":
            df_prep[c] = pd.to_numeric(df_prep[c], errors="coerce")

        df_prep[c] = pd.to_numeric(df_prep[c], errors="coerce")
        df_prep.loc[~df_prep[c].isin([0, 1]) & df_prep[c].notna(), c] = np.nan
        df_prep[c] = df_prep[c].astype("Int64")

    # Categorical
    for c in categorical_cols:
        df_prep[c] = df_prep[c].astype("category")

    # Numeric
    for c in numeric_cols:
        df_prep[c] = pd.to_numeric(df_prep[c], errors="coerce")

    # ---------------------------
    # 3) PLAUSIBILITY RULES
    # ---------------------------
    rules = {
        "age": lambda s: (s >= 18) & (s <= 100),
        "income": lambda s: s >= 0,
        "years_at_job": lambda s: s >= 0,
        "years_at_address": lambda s: s >= 0,
        "loan_amount": lambda s: s > 0,
        "dti": lambda s: (s >= 0) & (s <= 5),
        "num_inquiries_6m": lambda s: s >= 0,
        "num_delinquencies_24m": lambda s: s >= 0,
        "num_collections": lambda s: s >= 0,
        "card_utilization": lambda s: (s >= 0) & (s <= 2),
    }

    for col, rule in rules.items():
        if col in df_prep.columns:
            mask = df_prep[col].notna()
            df_prep.loc[mask & (~rule(df_prep[col])), col] = np.nan

    # ---------------------------
    # 4) MISSING VALUE HANDLING
    # ---------------------------
    excluded = set(target_cols)
    X_num = [c for c in numeric_cols if c not in excluded]
    X_cat = [c for c in categorical_cols if c not in excluded]
    X_bin = [c for c in binary_cols if c not in excluded and c != "default_12m"]

    num_missing_indicators = []

    # Numeric: median + optional missing indicator
    for c in X_num:
        if df_prep[c].isna().any():
            if add_missing_indicators:
                ind = f"{c}__missing"
                df_prep[ind] = df_prep[c].isna().astype(int)
                num_missing_indicators.append(ind)
            median = df_prep[c].median(skipna=True)
            df_prep[c] = df_prep[c].fillna(median)

    # Categorical: add explicit "missing"
    for c in X_cat:
        if df_prep[c].isna().any():
            df_prep[c] = df_prep[c].cat.add_categories(["missing"]).fillna("missing")

    # Binary: mode imputation
    for c in X_bin:
        if df_prep[c].isna().any():
            mode = df_prep[c].mode(dropna=True)
            fill = int(mode.iloc[0]) if len(mode) > 0 else 0
            df_prep[c] = df_prep[c].fillna(fill).astype(int)

    # ---------------------------
    # 5) WINSORIZATION
    # ---------------------------
    winsor_info = []
    for c in X_num:
        if df_prep[c].notna().sum() >= 30:
            lo = df_prep[c].quantile(winsor_limits[0])
            hi = df_prep[c].quantile(winsor_limits[1])
            df_prep[c] = df_prep[c].clip(lo, hi)
            winsor_info.append((c, lo, hi))

    # ---------------------------
    # 6) OPTIONAL LOG FEATURES
    # ---------------------------
    skewed_added = []
    log_features = []

    if add_log_features:
        for c in ["income", "loan_amount"]:
            if c in X_num and (df_prep[c] > 0).all():
                skew = df_prep[c].skew()
                if pd.notna(skew) and skew > skew_threshold:
                    newc = f"log1p_{c}"
                    df_prep[newc] = np.log1p(df_prep[c])
                    log_features.append(newc)
                    skewed_added.append((c, float(skew)))

    # ---------------------------
    # 7) ONE-HOT ENCODING
    # ---------------------------
    df_X = df_prep[X_num + X_bin + num_missing_indicators + log_features].copy()

    if X_cat:
        df_cat = pd.get_dummies(df_prep[X_cat], drop_first=True, dtype=int)
        df_X = pd.concat([df_X, df_cat], axis=1)

    # ---------------------------
    # 8) FINAL ASSEMBLY
    # ---------------------------
    df_model = df_X.copy()

    for t in target_cols:
        df_model[t] = df_prep[t]

    # Final predictor check
    predictor_cols = df_X.columns.tolist()
    if df_model[predictor_cols].isna().any().any():
        raise ValueError("Missing values remain in predictors after preprocessing.")

    if verbose:
        print("Data preparation complete.")
        print(f"Rows: {df.shape[0]}")
        print(f"Predictors: {len(predictor_cols)}")
        print(f"Targets: {target_cols}")

    metadata = {
        "numeric_predictors": X_num,
        "categorical_predictors": X_cat,
        "binary_predictors": X_bin,
        "missing_indicators": num_missing_indicators,
        "winsorization": winsor_info,
        "log_features": log_features,
    }

    return {
        "df_model": df_model,
        "predictors": predictor_cols,
        "targets": target_cols,
        "metadata": metadata
    }


In [ ]:
# Place to call your function and creating modeling-ready datasets
dfDevelopment = prepare_credit_data(df_100)["df_model"] 
dfHoldout = prepare_credit_data(df_25)["df_model"]

# Overfitting demonstrated
The lecture introduced you to the *fundamental problem of overfitting*, claiming that decision trees are particularly vulnerable to overfitting to make a case for tree pruning. 

<p align="center">
  <img src="https://raw.githubusercontent.com/Humboldt-WI/demopy/main/tree_pruning.png" alt="Overfitting in decision trees" width="480">
</p>

## Task:

Verify the illustrated relationship between tree depth and overfitting. To that end, your task is to:
1. Partition the data of `dfDevelopment` into a training and validation set using ratios of 80%/20%.
2. Train a decision tree classifier on the training data with varying tree depth (e.g., from 1 to 10).
3. Evaluate the performance of each model on both the training and validation sets using an appropriate metric.
4. Plot the training and validation performance against tree depth to visualize the overfitting phenomenon.

The coding tasks asks for a basic ML pipeline involving functions/classes `train_test_split`, `DecisionTreeClassifier`, amongst others. It is a good exercise to code the solution yourself, but you can also use AI; a lazy prompt should do, no discussion of a prompting strategy warranted.

In [7]:
# Place for your solution

# Benchmarking
We finally hit the main part of the practice session: benchmarking different machine learning algorithms for credit risk modeling. This exercises addresses a common questions in machine learning applications: *which algorithm is best suited for my data?*
Research papers that introduce novel methodologies also include benchmarking exercises to demonstrate the superiority of their method. Lastly, a learning goal of this exercise is to familiarize you with standard machine learning workflows and relevant Python functions/classes. 

We frame the benchmarking task such that it is realistic and perhaps a bit challenging. We discuss details in class, including our prompting strategy for the coding part. 

## Task: 
Compare several established machine learning algorithms using the development sample (i.e., `dfDevelopment`). Consider logistic regression (LR), neural networks (NN), decision trees (DT), random forests (RF), and extreme gradient boosting (XGB). For algorithms that exhibit hyperparameters, make sure these are properly tuned. Assess the performance of the different algorithms in terms of i) the area under the ROC curve, ii) the area under the prediction-recall curve, iii) the F-score, and iv) the Brier score (i.e., the mean squared error of the predicted probabilities compared to a zero-one coded target). Produce an estimate of model performance for each algorithm on a suitably selected subset of the development data. Also assess models on the holdout data (i.e., `dfHoldout`). Plot the ROC-curve, the PR-curve, and the confusion matrix for logistic regression and the overall best-performing model (if different) on the holdout sample.  

In [ ]:
# Place for your benchmarking code